# Partie 3 - Application Web Flask

**Backend :** Python Flask | **Frontend :** HTML5 + CSS3 + JavaScript

**Fonctionnalites :**
- Saisir un texte pour detecter FAKE / REAL
- Uploader une image pour detecter IA ou Reelle
- Interface sombre moderne avec barres de probabilite animees


---
## Etape 1 - Installation

---
## Etape 2 - Generer app.py (backend Flask)

In [ ]:
app_code = "\nimport os, json, numpy as np, joblib, re, string\nfrom flask import Flask, request, jsonify, render_template\nfrom PIL import Image\nimport io\n\nimport nltk\nfrom nltk.corpus import stopwords\nfrom nltk.stem import WordNetLemmatizer\nnltk.download('stopwords', quiet=True)\nnltk.download('wordnet',   quiet=True)\n\nimport tensorflow as tf\nfrom tensorflow import keras\n\napp = Flask(__name__, template_folder='templates', static_folder='static')\napp.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024\n\nprint('Chargement des modeles...')\nnlp_model      = joblib.load('models/best_classifier.pkl')\nnlp_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')\nprint('  NLP : OK')\n\nimage_model = keras.models.load_model('models/image_classifier_final.keras')\nwith open('models/class_names.json') as f:\n    idx_to_class = json.load(f)\nIMG_SIZE = (64, 64)\nprint('  Image : OK')\n\nlemmatizer = WordNetLemmatizer()\nstop_words  = set(stopwords.words('english'))\n\ndef preprocess_text(text):\n    if not isinstance(text, str): return ''\n    text = text.lower()\n    text = re.sub(r'http\\S+|www\\S+', '', text)\n    text = re.sub(r'[^a-z ]', ' ', text)\n    tokens = text.split()\n    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]\n    tokens = [lemmatizer.lemmatize(w) for w in tokens]\n    return ' '.join(tokens)\n\n@app.route('/')\ndef index():\n    return render_template('index.html')\n\n@app.route('/predict_text', methods=['POST'])\ndef predict_text():\n    data = request.get_json()\n    if not data or 'text' not in data or len(data['text'].strip()) < 10:\n        return jsonify({'error': 'Texte trop court (minimum 10 caracteres)'}), 400\n    text     = data['text'].strip()\n    cleaned  = preprocess_text(text)\n    features = nlp_vectorizer.transform([cleaned])\n    pred     = nlp_model.predict(features)[0]\n    label    = 'REAL' if pred == 1 else 'FAKE'\n    try:\n        proba      = nlp_model.predict_proba(features)[0]\n        real_proba = proba[1] * 100\n        fake_proba = proba[0] * 100\n    except AttributeError:\n        score      = float(nlp_model.decision_function(features)[0])\n        real_proba = min(50 + score * 15, 99.9) if score > 0 else max(50 + score * 15, 0.1)\n        fake_proba = 100 - real_proba\n    return jsonify({'label': label,\n                    'confidence': round(max(real_proba, fake_proba), 1),\n                    'real_proba': round(real_proba, 1),\n                    'fake_proba': round(fake_proba, 1)})\n\n@app.route('/predict_image', methods=['POST'])\ndef predict_image():\n    if 'image' not in request.files:\n        return jsonify({'error': 'Aucune image envoyee'}), 400\n    file = request.files['image']\n    if file.filename == '':\n        return jsonify({'error': 'Nom de fichier vide'}), 400\n    allowed = {'png','jpg','jpeg','webp','bmp'}\n    ext = file.filename.rsplit('.', 1)[-1].lower()\n    if ext not in allowed:\n        return jsonify({'error': 'Format non supporte'}), 400\n    img  = Image.open(io.BytesIO(file.read())).convert('RGB')\n    img  = img.resize(IMG_SIZE)\n    arr  = np.array(img) / 255.0\n    arr  = np.expand_dims(arr, axis=0)\n    proba      = float(image_model.predict(arr, verbose=0)[0][0])\n    label      = 'REAL' if proba >= 0.5 else 'FAKE'\n    real_proba = proba * 100\n    fake_proba = (1 - proba) * 100\n    return jsonify({'label': label,\n                    'confidence': round(max(real_proba, fake_proba), 1),\n                    'real_proba': round(real_proba, 1),\n                    'fake_proba': round(fake_proba, 1)})\n\nif __name__ == '__main__':\n    app.run(debug=True, port=5000)\n"
import os

os.makedirs('web_app/templates', exist_ok=True)
os.makedirs('web_app/static',    exist_ok=True)
with open('web_app/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())
print('web_app/app.py cree')

---
## Etape 3 - Generer index.html (interface)

In [ ]:
html_code = '<!DOCTYPE html>\n<html lang="fr">\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <title>TruthLens - Fake News & AI Image Detector</title>\n  <link rel="stylesheet" href="/static/style.css">\n</head>\n<body>\n  <header>\n    <div class="header-inner">\n      <div class="logo">\n        <span class="logo-icon">&#128269;</span>\n        <span class="logo-text">TruthLens</span>\n      </div>\n      <p class="header-sub">Fake News Detection &amp; AI Image Analysis</p>\n    </div>\n  </header>\n\n  <main>\n    <div class="tabs">\n      <button class="tab-btn active" onclick="switchTab(\'text\')">&#128240; Fake News Detection</button>\n      <button class="tab-btn" onclick="switchTab(\'image\')">&#128444; AI Image Detection</button>\n    </div>\n\n    <section id="tab-text" class="tab-panel active">\n      <div class="card">\n        <h2>Analyser un article de presse</h2>\n        <p class="hint">Colle un article ou un extrait (minimum 10 mots) &bull; Ctrl+Enter pour analyser</p>\n        <textarea id="news-input" placeholder="Ex: Scientists discover that the Earth is flat and NASA has been hiding this for decades..." rows="7"></textarea>\n        <div class="btn-row">\n          <button class="btn-primary" onclick="analyzeText()">Analyser l\'article</button>\n          <button class="btn-secondary" onclick="clearText()">Effacer</button>\n        </div>\n        <div id="text-loader" class="loader hidden">Analyse en cours...</div>\n        <div id="text-result" class="result-box hidden"></div>\n      </div>\n      <div class="card examples-card">\n        <h3>Exemples rapides</h3>\n        <div class="examples-grid">\n          <div class="example-chip fake" onclick="setExample(\'fake1\')">Fake : complot gouvernemental</div>\n          <div class="example-chip real" onclick="setExample(\'real1\')">Real : article economique</div>\n          <div class="example-chip fake" onclick="setExample(\'fake2\')">Fake : sante / remede cache</div>\n          <div class="example-chip real" onclick="setExample(\'real2\')">Real : decouverte scientifique</div>\n        </div>\n      </div>\n    </section>\n\n    <section id="tab-image" class="tab-panel hidden">\n      <div class="card">\n        <h2>Detection d\'image generee par IA</h2>\n        <p class="hint">Upload une image JPG, PNG ou WEBP (max 16 MB)</p>\n        <div class="drop-zone" id="drop-zone"\n             onclick="document.getElementById(\'img-input\').click()"\n             ondragover="dragOver(event)" ondragleave="dragLeave(event)" ondrop="dropImage(event)">\n          <div class="drop-icon">&#128444;</div>\n          <p>Glisse une image ici ou <strong>clique pour choisir</strong></p>\n          <p class="drop-hint">JPG &bull; PNG &bull; WEBP &bull; BMP</p>\n        </div>\n        <input type="file" id="img-input" accept="image/*" onchange="handleImageUpload(event)" style="display:none">\n        <div id="img-preview-container" class="hidden">\n          <img id="img-preview" src="" alt="preview">\n          <div class="btn-row" style="justify-content:center">\n            <button class="btn-primary" onclick="analyzeImage()">Analyser l\'image</button>\n            <button class="btn-secondary" onclick="clearImage()">Effacer</button>\n          </div>\n        </div>\n        <div id="image-loader" class="loader hidden">Analyse en cours...</div>\n        <div id="image-result" class="result-box hidden"></div>\n      </div>\n    </section>\n  </main>\n\n  <footer>\n    <p>Projet ML &mdash; NLP (TF-IDF + SVM) &amp; Deep Learning (CNN / MobileNetV2)</p>\n  </footer>\n  <script src="/static/app.js"></script>\n</body>\n</html>'

with open('web_app/templates/index.html', 'w', encoding='utf-8') as f:
    f.write(html_code.strip())
print('web_app/templates/index.html cree')

---
## Etape 4 - Generer style.css

In [6]:
css_code = '\n* { box-sizing: border-box; margin: 0; padding: 0; }\n:root {\n  --bg: #0f1117; --surface: #1a1d27; --surface2: #22263a;\n  --border: #2e3250; --accent: #6c5ce7; --accent2: #a29bfe;\n  --green: #00b894; --red: #e17055; --text: #e8e8f0;\n  --muted: #8888aa; --radius: 14px;\n}\nbody { font-family: "Segoe UI", system-ui, sans-serif; background: var(--bg); color: var(--text); min-height: 100vh; display: flex; flex-direction: column; }\nheader { background: #16192b; border-bottom: 1px solid var(--border); padding: 1.5rem 2rem; }\n.header-inner { max-width: 800px; margin: 0 auto; }\n.logo { display: flex; align-items: center; gap: 10px; margin-bottom: 4px; }\n.logo-icon { font-size: 28px; }\n.logo-text  { font-size: 24px; font-weight: 700; color: var(--accent2); }\n.header-sub { color: var(--muted); font-size: 14px; }\nmain { flex: 1; max-width: 800px; width: 100%; margin: 0 auto; padding: 2rem 1.5rem; }\n.tabs { display: flex; gap: 8px; margin-bottom: 1.5rem; border-bottom: 1px solid var(--border); }\n.tab-btn { background: none; border: none; color: var(--muted); padding: 10px 20px; font-size: 15px; cursor: pointer; border-bottom: 2px solid transparent; transition: all .2s; border-radius: 8px 8px 0 0; }\n.tab-btn:hover { color: var(--text); }\n.tab-btn.active { color: var(--accent2); border-bottom-color: var(--accent); }\n.tab-panel { display: none; }\n.tab-panel.active { display: block; }\n.card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 1.75rem; margin-bottom: 1.25rem; }\n.card h2 { font-size: 18px; margin-bottom: 6px; }\n.card h3 { font-size: 15px; margin-bottom: 12px; color: var(--muted); }\n.hint { color: var(--muted); font-size: 13px; margin-bottom: 14px; }\ntextarea { width: 100%; background: var(--surface2); border: 1px solid var(--border); border-radius: 10px; color: var(--text); font-size: 14px; padding: 14px; resize: vertical; transition: border-color .2s; font-family: inherit; }\ntextarea:focus { outline: none; border-color: var(--accent); }\n.btn-row { display: flex; gap: 10px; margin-top: 14px; }\n.btn-primary, .btn-secondary { padding: 10px 22px; border-radius: 9px; border: none; cursor: pointer; font-size: 14px; font-weight: 600; transition: all .2s; }\n.btn-primary { background: var(--accent); color: #fff; }\n.btn-primary:hover { background: #7d6ff0; transform: translateY(-1px); }\n.btn-secondary { background: var(--surface2); color: var(--muted); border: 1px solid var(--border); }\n.btn-secondary:hover { color: var(--text); }\n.result-box { margin-top: 18px; padding: 1.25rem 1.5rem; border-radius: 12px; border: 1px solid var(--border); animation: fadeIn .4s ease; }\n.result-box.real { border-color: var(--green); background: rgba(0,184,148,.08); }\n.result-box.fake { border-color: var(--red);   background: rgba(225,112,85,.08); }\n.result-header { display: flex; align-items: center; gap: 12px; margin-bottom: 14px; }\n.result-badge { font-size: 20px; font-weight: 800; padding: 4px 18px; border-radius: 8px; }\n.result-badge.real { background: rgba(0,184,148,.2); color: var(--green); }\n.result-badge.fake { background: rgba(225,112,85,.2); color: var(--red);  }\n.result-confidence { font-size: 14px; color: var(--muted); }\n.proba-row { display: flex; flex-direction: column; gap: 8px; }\n.proba-item label { display: flex; justify-content: space-between; font-size: 13px; margin-bottom: 4px; color: var(--muted); }\n.proba-track { height: 8px; border-radius: 4px; background: var(--surface2); overflow: hidden; }\n.proba-fill { height: 100%; border-radius: 4px; transition: width .8s cubic-bezier(.4,0,.2,1); }\n.proba-fill.real { background: var(--green); }\n.proba-fill.fake { background: var(--red); }\n.examples-grid { display: flex; flex-wrap: wrap; gap: 8px; }\n.example-chip { padding: 7px 14px; border-radius: 20px; font-size: 13px; cursor: pointer; transition: all .2s; border: 1px solid transparent; }\n.example-chip.fake { background: rgba(225,112,85,.12); color: var(--red);   border-color: rgba(225,112,85,.3); }\n.example-chip.real { background: rgba(0,184,148,.12);  color: var(--green); border-color: rgba(0,184,148,.3);  }\n.example-chip:hover { transform: translateY(-2px); opacity: .85; }\n.drop-zone { border: 2px dashed var(--border); border-radius: 14px; padding: 3rem 2rem; text-align: center; cursor: pointer; transition: all .2s; }\n.drop-zone:hover, .drop-zone.dragover { border-color: var(--accent); background: rgba(108,92,231,.06); }\n.drop-icon { font-size: 48px; margin-bottom: 12px; }\n.drop-zone p { color: var(--muted); font-size: 15px; }\n.drop-hint { font-size: 12px; margin-top: 6px; }\n#img-preview-container { margin-top: 1.5rem; display: flex; flex-direction: column; align-items: center; gap: 14px; }\n#img-preview { max-width: 300px; max-height: 300px; border-radius: 12px; border: 1px solid var(--border); object-fit: contain; }\n.loader { margin-top: 16px; text-align: center; color: var(--muted); font-size: 14px; padding: 12px; }\n.loader::before { content: ""; display: inline-block; width: 18px; height: 18px; border: 2px solid var(--border); border-top-color: var(--accent); border-radius: 50%; animation: spin .8s linear infinite; margin-right: 8px; vertical-align: middle; }\nfooter { text-align: center; padding: 1.25rem; color: var(--muted); font-size: 12px; border-top: 1px solid var(--border); }\n.hidden { display: none !important; }\n@keyframes fadeIn { from { opacity:0; transform:translateY(6px); } to { opacity:1; transform:none; } }\n@keyframes spin   { to { transform: rotate(360deg); } }\n@media (max-width: 600px) { main { padding: 1rem; } .btn-row { flex-direction: column; } }\n'

with open('web_app/static/style.css', 'w', encoding='utf-8') as f:
    f.write(css_code.strip())
print('web_app/static/style.css cree')

web_app/static/style.css cree


---
## Etape 5 - Generer app.js (logique frontend)

In [7]:
js_code = '\nconst EXAMPLES = {\n  fake1: "BREAKING: The president was secretly replaced by a robot in 2020. Anonymous sources inside the White House confirm this shocking global conspiracy that mainstream media refuses to cover.",\n  real1: "The Federal Reserve raised interest rates by 25 basis points on Wednesday following its latest two-day policy meeting, in line with expectations from Wall Street analysts.",\n  fake2: "Scientists discover that drinking bleach mixed with lemon juice cures all diseases including cancer. The government and Big Pharma have been hiding this simple cure for decades.",\n  real2: "Researchers at MIT have published new findings on quantum computing, demonstrating a 1000-qubit processor capable of solving optimization problems faster than classical computers."\n};\n\nlet selectedFile = null;\n\nfunction switchTab(tab) {\n  document.querySelectorAll(".tab-panel").forEach(p => { p.classList.remove("active"); p.classList.add("hidden"); });\n  document.querySelectorAll(".tab-btn").forEach(b => b.classList.remove("active"));\n  const panel = document.getElementById("tab-" + tab);\n  panel.classList.add("active");\n  panel.classList.remove("hidden");\n  document.querySelectorAll(".tab-btn")[tab === "text" ? 0 : 1].classList.add("active");\n}\n\nfunction show(id) { document.getElementById(id).classList.remove("hidden"); }\nfunction hide(id) { document.getElementById(id).classList.add("hidden"); }\n\nfunction buildResultHTML(data) {\n  const isReal = data.label === "REAL";\n  const cls    = isReal ? "real" : "fake";\n  const icon   = isReal ? "OK" : "FAKE";\n  const msg    = data.type === "text"\n    ? (isReal ? "Cet article semble authentique." : "Cet article ressemble a une fake news.")\n    : (isReal ? "Cette image semble reelle." : "Cette image semble generee par IA.");\n  return \'<div class="result-header">\'\n       + \'<span class="result-badge \' + cls + \'">\' + icon + \'</span>\'\n       + \'<span class="result-confidence">Confiance : <strong>\' + data.confidence + \'%</strong></span>\'\n       + \'</div>\'\n       + \'<p style="font-size:14px;color:var(--muted);margin-bottom:14px;">\' + msg + \'</p>\'\n       + \'<div class="proba-row">\'\n       + \'<div class="proba-item"><label><span>REAL</span><span>\' + data.real_proba + \'%</span></label>\'\n       + \'<div class="proba-track"><div class="proba-fill real" style="width:\' + data.real_proba + \'%"></div></div></div>\'\n       + \'<div class="proba-item"><label><span>FAKE</span><span>\' + data.fake_proba + \'%</span></label>\'\n       + \'<div class="proba-track"><div class="proba-fill fake" style="width:\' + data.fake_proba + \'%"></div></div></div>\'\n       + \'</div>\';\n}\n\nasync function analyzeText() {\n  const text = document.getElementById("news-input").value.trim();\n  if (text.length < 10) { alert("Minimum 10 caracteres requis."); return; }\n  hide("text-result");\n  show("text-loader");\n  try {\n    const res  = await fetch("/predict_text", { method:"POST", headers:{"Content-Type":"application/json"}, body: JSON.stringify({text}) });\n    const data = await res.json();\n    if (data.error) { alert("Erreur : " + data.error); }\n    else {\n      data.type = "text";\n      const box = document.getElementById("text-result");\n      box.className = "result-box " + (data.label === "REAL" ? "real" : "fake");\n      box.innerHTML = buildResultHTML(data);\n      show("text-result");\n    }\n  } catch(e) { alert("Erreur de connexion au serveur."); }\n  finally    { hide("text-loader"); }\n}\n\nfunction clearText()         { document.getElementById("news-input").value = ""; hide("text-result"); }\nfunction setExample(key)     { document.getElementById("news-input").value = EXAMPLES[key]; hide("text-result"); }\n\nfunction handleImageUpload(event) {\n  const file = event.target.files[0];\n  if (!file) return;\n  selectedFile = file;\n  document.getElementById("img-preview").src = URL.createObjectURL(file);\n  show("img-preview-container");\n  hide("image-result");\n}\n\nfunction dragOver(e)  { e.preventDefault(); document.getElementById("drop-zone").classList.add("dragover"); }\nfunction dragLeave(e) { document.getElementById("drop-zone").classList.remove("dragover"); }\nfunction dropImage(e) {\n  e.preventDefault();\n  document.getElementById("drop-zone").classList.remove("dragover");\n  const file = e.dataTransfer.files[0];\n  if (!file) return;\n  selectedFile = file;\n  document.getElementById("img-preview").src = URL.createObjectURL(file);\n  show("img-preview-container");\n  hide("image-result");\n}\n\nasync function analyzeImage() {\n  if (!selectedFile) { alert("Aucune image selectionnee."); return; }\n  hide("image-result");\n  show("image-loader");\n  const formData = new FormData();\n  formData.append("image", selectedFile);\n  try {\n    const res  = await fetch("/predict_image", { method:"POST", body: formData });\n    const data = await res.json();\n    if (data.error) { alert("Erreur : " + data.error); }\n    else {\n      data.type = "image";\n      const box = document.getElementById("image-result");\n      box.className = "result-box " + (data.label === "REAL" ? "real" : "fake");\n      box.innerHTML = buildResultHTML(data);\n      show("image-result");\n    }\n  } catch(e) { alert("Erreur de connexion au serveur."); }\n  finally    { hide("image-loader"); }\n}\n\nfunction clearImage() {\n  selectedFile = null;\n  document.getElementById("img-preview").src = "";\n  document.getElementById("img-input").value = "";\n  hide("img-preview-container");\n  hide("image-result");\n}\n\ndocument.addEventListener("keydown", function(e) {\n  if ((e.ctrlKey || e.metaKey) && e.key === "Enter") {\n    if (document.getElementById("tab-text").classList.contains("active")) analyzeText();\n  }\n});\n'

with open('web_app/static/app.js', 'w', encoding='utf-8') as f:
    f.write(js_code.strip())
print('web_app/static/app.js cree')

web_app/static/app.js cree


---
## Etape 6 - Verification des fichiers

In [8]:
import os

files_to_check = [
    'web_app/app.py',
    'web_app/templates/index.html',
    'web_app/static/style.css',
    'web_app/static/app.js',
    'models/best_classifier.pkl',
    'models/tfidf_vectorizer.pkl',
    'models/image_classifier_final.keras',
    'models/class_names.json',
]

all_ok = True
print("Verification des fichiers :")
for fpath in files_to_check:
    exists = os.path.exists(fpath)
    status = "OK     " if exists else "MANQUANT"
    size   = f"({os.path.getsize(fpath):,} bytes)" if exists else ""
    if not exists: all_ok = False
    print(f"  [{status}] {fpath} {size}")
print()
print("Tous les fichiers OK - Pret a lancer Flask." if all_ok else "Certains fichiers manquent.")

Verification des fichiers :
  [OK     ] web_app/app.py (3,542 bytes)
  [OK     ] web_app/templates/index.html (3,601 bytes)
  [OK     ] web_app/static/style.css (5,604 bytes)
  [OK     ] web_app/static/app.js (5,624 bytes)
  [OK     ] models/best_classifier.pkl (400,731 bytes)
  [OK     ] models/tfidf_vectorizer.pkl (2,048,436 bytes)
  [OK     ] models/image_classifier_final.keras (3,994,414 bytes)
  [OK     ] models/class_names.json (26 bytes)

Tous les fichiers OK - Pret a lancer Flask.


---
## Etape 7 - Lancer l'application

In [16]:
# Option A - depuis le notebook (thread)
import threading, time

def run_flask():
    import subprocess, sys, os
    os.chdir('web_app')
    subprocess.run([sys.executable, 'app.py'])

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(3)
print("Serveur Flask demarre !")
print("Ouvre : http://127.0.0.1:5000")

Exception in thread Thread-8 (run_flask):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/js/fbjrnms50q16t06myxq4l5l80000gn/T/ipykernel_39213/1115301250.py", line 6, in run_flask
FileNotFoundError: [Errno 2] No such file or directory: 'web_app'


Serveur Flask demarre !
Ouvre : http://127.0.0.1:5000


---
## Recapitulatif - Endpoints API

| Endpoint | Methode | Input | Output |
|----------|---------|-------|--------|
| `/` | GET | - | Page HTML |
| `/predict_text` | POST | JSON `{"text": "..."}` | `{label, confidence, real_proba, fake_proba}` |
| `/predict_image` | POST | FormData `image` | `{label, confidence, real_proba, fake_proba}` |

**Test rapide (terminal) :**
```bash
# Texte
curl -X POST http://127.0.0.1:5000/predict_text \
     -H "Content-Type: application/json" \
     -d '{"text": "The government is hiding the cure for cancer to protect Big Pharma profits."}'

# Image
curl -X POST http://127.0.0.1:5000/predict_image \
     -F "image=@/chemin/vers/image.jpg"
```

**Partie 3 terminee ! Pret pour la structure complete du projet.**
